In [ ]:
import sys
sys.path.insert(0, '../..')

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import edunn as nn
from edunn import utils

In [ ]:
np.set_printoptions(threshold=sys.maxsize)

# Batch Normalization Layer

Batch normalization addresses the problem of poor initialization in neural networks. It can be understood as performing a preprocessing step at every layer of the network. It forces the activations in a network to adopt a unit Gaussian distribution at the beginning of training. This ensures that every neuron has approximately the same output distribution across the network, and it improves the convergence rate.

Explaining why the distribution of the activations in a network matters is beyond the scope of these guides, but if you are interested you can refer to pages 46 — 62 of the [lecture slides](http://cs231n.stanford.edu/slides/2019/cs231n_2019_lecture07.pdf) offered by Stanford University's course.

## Forward Method

Suppose we have a batch of activations $x$ at a layer; the zero-mean, unit-variance version of $x$, $\hat{x}$, is:

$$\hat{x}^{(k)}=\frac{x^{(k)}-\mathbb{E}[x^{(k)}]}{\sqrt{\text{Var}[x^{(k)}]}}$$

This is in fact a differentiable operation, which is why we can apply batch normalization during training.

Computing it boils down to calculating the mean $\mu_\mathcal{B}$ and the variance $\sigma_\mathcal{B}^2$ of a mini-batch $\mathcal{B}=\{x_1, \dots, x_N\}$. The layer's learnable parameters are $\gamma$ and $\beta$, which are used to scale and shift the normalized values.

$$
\begin{aligned}
\mu_\mathcal{B} &= \frac{1}{N} \sum_{i=1}^{N} x_i & \text{(mini-batch mean)} \\
\sigma_\mathcal{B}^2 &= \frac{1}{N} \sum_{i=1}^{N} (x_i - \mu_\mathcal{B})^2 & \text{(mini-batch variance)} \\
\hat{x}_i &= \frac{x_i - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2+\epsilon}} & \text{(normalize)} \\
\text{\textbf{BN}}_{\gamma,\beta}(x_i) &\stackrel{\text{def}}{=} \gamma \hat{x}_i + \beta = y_i & \text{(scale and shift)} 
\end{aligned}
$$

> NOTE: In the implementation, we insert the `BatchNorm` layer right after a `Dense` layer or a `Conv2d` layer, and before the nonlinear layers.


In [ ]:
np.random.seed(123)

din=10
batch_size=2

x = np.random.rand(batch_size,din)
w = np.random.rand(din)
b = np.random.rand(din)
                       
gamma_initializer = nn.initializers.Constant(w)
beta_initializer = nn.initializers.Constant(b)

layer=nn.BatchNorm(num_features=din, gamma_initializer=gamma_initializer, beta_initializer=beta_initializer)

In [ ]:
x

In [ ]:
y = layer.forward(x)
y, y.shape

## Backward Method

### `δEδβ`

The gradients of the error $E$ with respect to the parameter $\beta$ can be computed by taking partial derivatives, as explained in previous guides:

$$
\frac{\partial E}{\partial \beta} = \frac{\partial E}{\partial y} \cdot \frac{\partial y}{\partial \beta}
$$

Since $y$ is a vector of $N$ elements, we have to sum over all of its values in order to apply the chain rule:

$$
\frac{\partial E}{\partial \beta} = \frac{\partial E}{\partial y_1} \cdot \frac{\partial y_1}{\partial \beta} + \cdots + \frac{\partial E}{\partial y_N} \cdot \frac{\partial y_N}{\partial \beta} 
\qquad \text{where} \qquad
\frac{\partial y_i}{\partial \beta} = \frac{\partial (\gamma \hat{x}_i + \beta)}{\partial \beta} = 1
$$

and so:

$$
\frac{\partial E}{\partial \beta} = \sum\limits_{i=1}^N \frac{\partial E}{\partial y_i} \cdot 1
$$

### `δEδγ`

The gradients of the error $E$ with respect to the parameter $\gamma$ can be computed by taking partial derivatives, as explained in previous guides:

$$
\frac{\partial E}{\partial \gamma} = \frac{\partial E}{\partial y} \cdot \frac{\partial y}{\partial \gamma}
$$

Since $y$ is a vector of $N$ elements, we have to sum over all of its values in order to apply the chain rule:

$$
\frac{\partial E}{\partial \gamma} = \frac{\partial E}{\partial y_1} \cdot \frac{\partial y_1}{\partial \gamma} + \cdots + \frac{\partial E}{\partial y_N} \cdot \frac{\partial y_N}{\partial \gamma} 
\qquad \text{where} \qquad
\frac{\partial y_i}{\partial \gamma} = \frac{\partial (\gamma \hat{x}_i + \beta)}{\partial \gamma} = \hat{x}_i
$$

and so:

$$
\frac{\partial E}{\partial \gamma} = \sum\limits_{i=1}^N \frac{\partial E}{\partial y_i} \cdot \hat{x}_i
$$

### `δEδx`

<!-- Using the chain rule from differential calculus, it tells us that the derivative of a composite function is the product of the derivatives of the functions that make it up. -->

Taking into account what each function depends on:

<center>

||||
|:-:|:-:|:-:|
|$E(y)$|$y(\hat{x},\gamma,\beta)$|$\hat{x}(\mu,\sigma^2,x)$|

</center>

We obtain:

$$
\dfrac{\partial E}{\partial x_i} = \frac{\partial E}{\partial \hat{x}_i} \cdot \frac{\partial \hat{x}_i}{\partial x_i} + \frac{\partial E}{\partial \mu} \cdot \frac{\partial \mu}{\partial x_i} + \frac{\partial E}{\partial \sigma^2} \cdot \frac{\partial \sigma^2}{\partial x_i}
$$

In the following subsections we will compute the corresponding expression for the gradient of each component.

#### `δEδx̂`

$$
\frac{\partial E}{\partial \hat{x}_i} = \frac{\partial E}{\partial y_i} \cdot \frac{\partial y_i}{\partial \hat{x}_i} = \frac{\partial E}{\partial y_i} \cdot \frac{\partial (\gamma \hat{x}_i + \beta)}{\partial \hat{x}_i} = \frac{\partial E}{\partial y_i} \cdot \gamma
$$

#### `δEδμ`

Note that $\sigma^2$ can be written as a function of $\mu$, and that is why $E$ depends on $\mu$ through two variables: $\hat{x}_i$​ and $\sigma^2$.

$$
\dfrac{\partial E}{\partial \mu} = \frac{\partial E}{\partial \hat{x}_i} \cdot \frac{\partial \hat{x}_i}{\partial \mu} + \frac{\partial E}{\partial \sigma^2} \cdot \frac{\partial \sigma^2}{\partial\mu}
$$

Computing the partial derivatives for `δx̂δμ` and `δσ²δμ`:

$$
\begin{aligned}
\hat{x}_i = \frac{(x_i - \mu)}{\sqrt{\sigma^2 + \epsilon}}
&\qquad \Rightarrow \qquad
\dfrac{\partial \hat{x}_i}{\partial \mu} = \frac{-1}{\sqrt{\sigma^2 + \epsilon}} \\
\sigma^2 = \frac{1}{N} \sum\limits_{i=1}^N (x_i - \mu)^2
&\qquad \Rightarrow \qquad
\dfrac{\partial \sigma^2}{\partial \mu} = \frac{1}{N} \sum\limits_{i=1}^N -2 \cdot (x_i - \mu) \\
\end{aligned}
$$

Substituting these last two expressions and keeping the gradients of the error $E$ as variables, we obtain:

$$
\begin{aligned}
\frac{\partial E}{\partial \mu} &= \bigg(\sum\limits_{i=1}^N  \frac{\partial E}{\partial \hat{x}_i} \cdot \frac{-1}{\sqrt{\sigma^2 + \epsilon}} \bigg) + \bigg( \frac{\partial E}{\partial \sigma^2} \cdot \frac{1}{N} \sum\limits_{i=1}^N -2(x_i - \mu)   \bigg) \qquad \\
&= \bigg(\sum\limits_{i=1}^N  \frac{\partial E}{\partial \hat{x}_i} \cdot \frac{-1}{\sqrt{\sigma^2 + \epsilon}} \bigg) + \bigg( \frac{\partial E}{\partial \sigma^2} \cdot (-2) \cdot \bigg( \frac{1}{N} \sum\limits_{i=1}^N x_i - \frac{1}{N} \sum\limits_{i=1}^N \mu   \bigg) \bigg) \qquad \\
&= \bigg(\sum\limits_{i=1}^N  \frac{\partial E}{\partial \hat{x}_i} \cdot \frac{-1}{\sqrt{\sigma^2 + \epsilon}} \bigg) + \bigg( \frac{\partial E}{\partial \sigma^2} \cdot (-2) \cdot \underbrace{\bigg( \mu - \frac{N \cdot \mu}{N} \bigg)}_{0} \bigg) \qquad \\
&= \sum\limits_{i=1}^N  \frac{\partial E}{\partial \hat{x}_i} \cdot \frac{-1}{\sqrt{\sigma^2 + \epsilon}} \qquad \\
\end{aligned}
$$

#### `δEδσ²`

$$\frac{\partial E}{\partial \sigma^2} = \frac{\partial E}{\partial \hat{x}} \cdot \frac{\partial \hat{x}}{\partial \sigma^2}$$

Rewriting $\hat{x}_i$ so that its derivative is easier to compute, we see that $(x_i - \mu)$ becomes a constant factor, so that:

$$
\hat{x}_i = (x_i - \mu)(\sigma^2 + \epsilon)^{-0.5}
\qquad \Rightarrow \qquad
\dfrac{\partial E}{\partial \sigma^2} = -0.5 \sum\limits_{i=1}^N \dfrac{\partial E}{\partial \hat{x}_i} (x_i - \mu) \cdot (\sigma^2 + \epsilon)^{-1.5}
$$

#### `δEδx` (cont.)

Computing the remaining partial derivatives (`δx̂δx`, `δμδx` and `δσ²δx`) of the original expression, we obtain:

<center>

||||
|:-:|:-:|:-:|
|$\dfrac{\partial \hat{x}_i}{\partial x_i} = \dfrac{1}{\sqrt{\sigma^2 + \epsilon}}$|$\dfrac{\partial \mu}{\partial x_i} = \dfrac{1}{N}$|$\dfrac{\partial \sigma^2}{\partial x_i} = \dfrac{2(x_i - \mu)}{N}$|

</center>

Finally, we can compute the gradient of the error $E$ with respect to $x$ using the following trick:

$$
(\sigma^2 + \epsilon)^{-1.5} = (\sigma^2 + \epsilon)^{-0.5}(\sigma^2 + \epsilon)^{-1} = (\sigma^2 + \epsilon)^{-0.5} \frac{1}{\sqrt{\sigma^2 + \epsilon}}\frac{1}{\sqrt{\sigma^2 + \epsilon}}
$$

and so:

$$
\begin{aligned}
\frac{\partial E}{\partial x_i} &= \bigg(\frac{\partial E}{\partial \hat{x}_i} \cdot \dfrac{1}{\sqrt{\sigma^2 + \epsilon}} \quad\; \bigg) + \bigg(\frac{\partial E}{\partial \mu} \cdot \dfrac{1}{N} \qquad\qquad\qquad\!\! \bigg) + \bigg(\frac{\partial E}{\partial \sigma^2} \cdot \dfrac{2(x_i - \mu)}{N}\bigg) \qquad \\
&= \bigg(\frac{\partial E}{\partial \hat{x}_i} \cdot \dfrac{1}{\sqrt{\sigma^2 + \epsilon}} \quad\; \bigg) + \bigg(\frac{1}{N} \sum\limits_{j=1}^N  \frac{\partial E}{\partial \hat{x}_j} \cdot \frac{-1}{\sqrt{\sigma^2 + \epsilon}}\bigg) + \bigg(-0.5 \sum\limits_{j=1}^N \frac{\partial E}{\partial \hat{x}_j} \cdot (x_j - \mu) \cdot (\sigma^2 + \epsilon)^{-1.5} \cdot \dfrac{2(x_i - \mu)}{N} \bigg) \qquad \\
&= \bigg(\frac{\partial E}{\partial \hat{x}_i} \cdot (\sigma^2 + \epsilon)^{-0.5} \bigg) - \bigg(\frac{(\sigma^2 + \epsilon)^{-0.5}}{N} \sum\limits_{j=1}^N  \frac{\partial E}{\partial \hat{x}_j} \;\, \bigg) - \bigg(\frac{(\sigma^2 + \epsilon)^{-0.5}}{N} \cdot \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \sum\limits_{j=1}^N \frac{\partial E}{\partial \hat{x}_j} \cdot \frac{(x_j - \mu)}{\sqrt{\sigma^2 + \epsilon}} \bigg )\qquad \\
&= \bigg(\frac{\partial E}{\partial \hat{x}_i} \cdot (\sigma^2 + \epsilon)^{-0.5} \bigg) - \bigg(\frac{(\sigma^2 + \epsilon)^{-0.5}}{N} \sum\limits_{j=1}^N  \frac{\partial E}{\partial \hat{x}_j} \;\, \bigg) - \bigg(\frac{(\sigma^2 + \epsilon)^{-0.5}}{N} \cdot \hat{x}_i \sum\limits_{j=1}^N \frac{\partial E}{\partial \hat{x}_j} \cdot \hat{x}_j \bigg )\qquad \\
&= \boxed{\frac{(\sigma^2 + \epsilon)^{-0.5}}{N} \bigg [N \frac{\partial E}{\partial \hat{x}_i} - \sum\limits_{j=1}^N  \frac{\partial E}{\partial \hat{x}_j} - \hat{x}_i \sum\limits_{j=1}^N \frac{\partial E}{\partial \hat{x}_j} \cdot \hat{x}_j\bigg ]} \qquad \\
&= \frac{(\sigma^2 + \epsilon)^{-0.5}}{N} \bigg [N \frac{\partial E}{\partial y_i} \cdot \gamma - \sum\limits_{j=1}^N  \frac{\partial E}{\partial y_j} \cdot \gamma - \hat{x}_i \sum\limits_{j=1}^N \frac{\partial E}{\partial y_j} \cdot \gamma \cdot \hat{x}_j\bigg ] \qquad \\
\end{aligned}
$$


In [ ]:
# Define the gradient of the output
g = np.random.rand(*y.shape)

# Propagate the gradient backward through the convolution
layer_grad = layer.backward(g)
layer_grad

## Checks with PyTorch

In [ ]:
import torch
import torch.nn as tnn

w = torch.from_numpy(w).to(torch.float)
b = torch.from_numpy(b).to(torch.float)
x = torch.from_numpy(x).to(torch.float)

batch_norm = tnn.BatchNorm1d(din)
batch_norm.weight.data = w
batch_norm.bias.data = b

x.requires_grad = True
batch_norm.weight.requires_grad = True
batch_norm.bias.requires_grad = True

y_torch = batch_norm(x)
y_torch

In [ ]:
utils.check_same(y_torch.detach().numpy(),y,tol=1e-5)

In [ ]:
# Define the gradient of the output
g = torch.from_numpy(g).to(torch.double)

# Propagate the gradient backward
x.grad = None  # Clear the existing gradients
y_torch.backward(g)

# Print the gradient of the input image, gamma and beta
print("Gradient of the input (δE/δx):")
print(x.grad, x.grad.shape)
print("\nGradient of gamma (δE/δγ):")
print(batch_norm.weight.grad, batch_norm.weight.grad.shape)
print("\nGradient of beta (δE/δβ):")
print(batch_norm.bias.grad, batch_norm.bias.grad.shape)

# Print the parameters of the BatchNorm layer
print("\nParameters of the BatchNorm layer:")
for name, param in batch_norm.named_parameters():
    print(name, param.data)

In [ ]:
utils.check_same(x.grad.numpy(),layer_grad[0],tol=1e-5)

In [ ]:
utils.check_same(batch_norm.weight.grad.numpy(),layer_grad[1]['w'],tol=1e-5)

In [ ]:
utils.check_same(batch_norm.bias.grad.numpy(),layer_grad[1]['b'],tol=1e-5)

In [ ]:
samples = 100
batch_size=8
din=10 # input dimension
input_shape=(batch_size,din)

# Check the derivatives of a BatchNorm model
# with random values for `w`, `b`, and `x`, the input
layer=nn.BatchNorm(din)


utils.check_gradient.common_layer(layer,input_shape,samples=samples,tolerance=1e-5)    